# 03 - Precision-Recall Curves

This notebook evaluates the EncryptionGuard ML model by computing and visualizing
precision-recall curves and PR-AUC scores on the test dataset.

In [ ]:
import sys
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add backend to path for imports
sys.path.insert(0, str(Path("../backend")))

# Configure plotting
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 7)

# Load the trained model
MODEL_PATH = Path("../backend/ml/artifacts/model.pkl")
with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

print(f"Model type: {type(model).__name__}")
print(f"Model loaded from: {MODEL_PATH}")

# Display model parameters if available
if hasattr(model, "get_params"):
    params = model.get_params()
    print(f"\nModel parameters:")
    for k, v in params.items():
        print(f"  {k}: {v}")

## Load Test Data

Using the project's `ml.train` module to load data and create train/test splits.

In [ ]:
from ml.train import load_data, create_splits

# Load and split data
X, y = load_data()
X_train, X_test, y_train, y_test = create_splits(X, y)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nFeature count: {X_train.shape[1]}")
print(f"Positive class ratio (test): {y_test.mean():.3f}")

## Compute Precision-Recall Curve

Generating precision-recall pairs for various probability thresholds.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

# Get predicted probabilities
if hasattr(model, "predict_proba"):
    y_scores = model.predict_proba(X_test)[:, 1]
elif hasattr(model, "decision_function"):
    y_scores = model.decision_function(X_test)
else:
    y_scores = model.predict(X_test)

# Compute precision-recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_scores)

# Compute PR-AUC (Average Precision)
pr_auc = average_precision_score(y_test, y_scores)

print(f"PR-AUC (Average Precision): {pr_auc:.4f}")
print(f"Number of thresholds: {len(thresholds)}")
print(f"\nPrecision range: [{precision.min():.4f}, {precision.max():.4f}]")
print(f"Recall range:    [{recall.min():.4f}, {recall.max():.4f}]")

## Precision-Recall Curve Plot

Visualizing the PR curve with the PR-AUC score annotated.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Plot PR curve
ax.plot(
    recall,
    precision,
    color="steelblue",
    linewidth=2,
    label=f"PR Curve (AUC = {pr_auc:.4f})"
)

# Fill area under curve
ax.fill_between(recall, precision, alpha=0.2, color="steelblue")

# Add baseline (prevalence of positive class)
baseline = y_test.mean()
ax.axhline(
    y=baseline,
    color="red",
    linestyle="--",
    linewidth=1,
    label=f"Baseline (prevalence = {baseline:.4f})"
)

# Annotate PR-AUC
ax.annotate(
    f"PR-AUC = {pr_auc:.4f}",
    xy=(0.6, 0.95),
    xycoords="axes fraction",
    fontsize=14,
    fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray")
)

ax.set_xlabel("Recall", fontsize=13)
ax.set_ylabel("Precision", fontsize=13)
ax.set_title("Precision-Recall Curve", fontsize=15)
ax.legend(loc="lower left", fontsize=12)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

plt.tight_layout()
plt.show()